# Aadhaar Data Preprocessing

This notebook prepares an analysis-ready **district × date** master panel by consolidating enrolment, demographic, and biometric datasets and integrating them on a common index.

**Keys and grain**
- Input grain: pincode × date → aggregated to district × date
- Join keys: `(date, state, district)`

**Pipeline (high level)**
- Consolidate split source files → one combined CSV per dataset
- Load combined tables, standardize keys/types, enforce numeric measures
- Aggregate to district × date: $$X_{d,t}=\sum_{p\in d} X_{p,t}$$
- Build skeleton $S=D\times T$ and LEFT JOIN each source (no row loss)
- Temporal alignment + controlled filling in the overlap window (when applicable)


### Base inputs: columns used downstream
Each base table contains identifiers plus measures. Downstream steps use only:
- **Common keys:** `date`, `state`, `district`
- **Enrolment measures:** `age_0_5`, `age_5_17`, `age_18_greater`
- **Demographic measures:** `demo_age_5_17`, `demo_age_17_`
- **Biometric measures:** `bio_age_5_17`, `bio_age_17_`

---

In [1]:
import pandas as pd
from pathlib import Path


### Combining enrolment data
df1 = pd.read_csv("data/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_0_500000.csv", parse_dates=["date"], dayfirst=True)
df2 = pd.read_csv("data/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_500000_1000000.csv", parse_dates=["date"], dayfirst=True)
df3 = pd.read_csv("data/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_1000000_1006029.csv", parse_dates=["date"], dayfirst=True)

# Stack rows
df = pd.concat([df1, df2, df3], ignore_index=True)

# Save
df.to_csv("data/combined/aadhar_enrolment_combined.csv", index=False)

### 0.2 Biometric files → single table
Stacks biometric source files and saves `data/combined/aadhar_biometric_combined.csv`.

In [2]:
### Combining biometric data

df1 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_0_500000.csv",parse_dates=["date"], dayfirst=True)
df2 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_500000_1000000.csv",parse_dates=["date"], dayfirst=True)
df3 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_1000000_1500000.csv",parse_dates=["date"], dayfirst=True)
df4 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_1500000_1861108.csv",parse_dates=["date"], dayfirst=True)

# Stack rows
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Save
df.to_csv("data/combined/aadhar_biometric_combined.csv", index=False)

## 1. Load combined datasets

In [3]:
# 1.1 Load the csv files
import pandas as pd

enroll = pd.read_csv("data/combined/aadhar_enrolment_combined.csv", parse_dates=["date"], dayfirst=False)
demo = pd.read_csv("data/combined/aadhar_demographic_combined.csv", parse_dates=["date"], dayfirst=False)
bio = pd.read_csv("data/combined/aadhar_biometric_combined.csv", parse_dates=["date"], dayfirst=False)


### 1.1 Date standardization
**What this cell does**
- Converts `date` to a strict datetime type
- Coerces invalid values to NaT so they can be detected
- Ensures downstream groupby/merge operations align correctly on time

In [4]:
# 1.3 Standardize date columns
for df in [enroll, demo, bio]:
    df["date"] = pd.to_datetime(df["date"], dayfirst=False, errors="coerce")


### 1.2 Date dtype validation
**What this cell does**
- Confirms each dataset’s `date` column is truly datetime
- Prevents silent merge failures from mismatched dtypes
- Acts as a quick sanity check before proceeding

In [5]:
print(pd.api.types.is_datetime64_any_dtype(enroll["date"]))
print(pd.api.types.is_datetime64_any_dtype(demo["date"]))
print(pd.api.types.is_datetime64_any_dtype(bio["date"]))

True
True
True


### 1.3 Numeric typing (measures)
**What this cell does**
- Defines the measure columns for each dataset
- Converts measures to numeric values (non-numeric → NaN)
- Guarantees aggregations (sums) behave mathematically

In [6]:
# 1.5 Ensure numeric columns are numeric

numeric_cols = {
    "enroll": ["age_0_5", "age_5_17", "age_18_greater"],
    "demo":   ["demo_age_5_17", "demo_age_17_"],
    "bio":    ["bio_age_5_17", "bio_age_17_"],
}

for col in numeric_cols["enroll"]:
    enroll[col] = pd.to_numeric(enroll[col], errors="coerce")

for col in numeric_cols["demo"]:
    demo[col] = pd.to_numeric(demo[col], errors="coerce")

for col in numeric_cols["bio"]:
    bio[col] = pd.to_numeric(bio[col], errors="coerce")


### 1.4 Export cleaned base tables (checkpoint)

In [7]:
enroll.to_csv("data/cleaned/clean_enrolment_raw.csv", index=False)
demo.to_csv("data/cleaned/clean_demographic_raw.csv", index=False)
bio.to_csv("data/cleaned/clean_biometric_raw.csv", index=False)


## 2. District level aggregation (daily)
### 2.1 Enrolment: pincode level → district totals
**What this cell does**
- Aggregates enrolment measures by `(date, state, district)`
- Uses additive consistency: $$X_{d,t}=\sum_{p\in d} X_{p,t}$$
- Produces `enroll_dist` at district × date granularity

**Why this matters**
- The final master table is district-based, so all sources must be aligned to the same unit of analysis.

In [8]:
# 2.1 Aggregate Aadhaat enrolment data

enroll_dist = (
    enroll
    .groupby(["date", "state", "district"], as_index=False)
    .agg({
        "age_0_5": "sum",
        "age_5_17": "sum",
        "age_18_greater": "sum"
    })
)

### 2.1.1 Aggregation sanity check (shapes)
Prints shapes before/after aggregation to confirm the granularity shift.

In [9]:
print(enroll.shape)
print(enroll_dist.shape)

(1006029, 7)
(66749, 6)


### 2.2 Demographic: district table preview (checkpoint)
Preview `demo_dist` (district × date demographic aggregates).

In [10]:
demo_dist.head()

NameError: name 'demo_dist' is not defined

### 2.3 Biometric updates: pincode level → district totals
**What this cell does**
- Aggregates biometric update measures by `(date, state, district)`
- Uses simple sums to preserve totals under aggregation
- Produces `bio_dist` at district × date granularity

In [ ]:
# 2.3 Aggregate Aadhaar biometric update data

bio_dist = (
    bio
    .groupby(["date", "state", "district"], as_index=False)
    .agg({
        "bio_age_5_17": "sum",
        "bio_age_17_": "sum"
    })
)

### 2.3.1 Aggregation sanity check (shapes)
**What this cell does**
- Prints row/column counts pre- and post-aggregation
- Confirms the groupby collapsed the raw table
- Validation only (no transformations)

In [ ]:
print(bio.shape)
print(bio_dist.shape)

(1861108, 6)
(79179, 5)


### 2.4 Totals preservation check (spot check)
**What this cell does**
- Selects one `(date, state, district)` key
- Compares raw-sum vs aggregated value for a representative column
- Confirms aggregation preserves totals

In [ ]:
# Checking if the totals are preserved

sample = enroll.query(
    "date == @enroll_dist.date.iloc[0] "
    "and state == @enroll_dist.state.iloc[0] "
    "and district == @enroll_dist.district.iloc[0]"
)

sample["age_0_5"].sum(), enroll_dist.iloc[0]["age_0_5"]


(np.int64(11), np.int64(11))

## 3. District Date skeleton (no row loss)
We construct a complete grid of all districts across the full date span.

**Key idea**
- District universe $D$ = unique `(state, district)` pairs
- Date universe $T$ = daily range covering the min/max across datasets

**Why this matters**
- Building the full index first makes missingness explicit and prevents accidental row loss during merges.

In [ ]:
districts.shape


(1095, 2)

### 3.2 Global date range
- Builds a daily `pd.date_range` to form the $T$ component

In [ ]:
# 3.2 Create the global date range

min_date = min(
    enroll_dist["date"].min(),
    demo_dist["date"].min(),
    bio_dist["date"].min()
)

max_date = max(
    enroll_dist["date"].max(),
    demo_dist["date"].max(),
    bio_dist["date"].max()
)

all_dates = pd.date_range(start=min_date, end=max_date, freq="D")


### 3.3 Skeleton construction (cartesian product)
**What this cell does**
- Creates the cartesian product of districts and dates
- Produces `skeleton` with one row per `(date, state, district)`

In [ ]:
# 3.3 Build the cartesian product (the skeleton)

skeleton = (
    districts
    .assign(key=1)
    .merge(
        pd.DataFrame({"date": all_dates, "key": 1}),
        on="key"
    )
    .drop("key", axis=1)
    .sort_values(["date", "state", "district"])
    .reset_index(drop=True)
)

### 3.4 Persist skeleton (checkpoint)

In [ ]:
skeleton.to_csv("data/district_date_skeleton.csv", index=False)


## 4. Integration (LEFT JOIN chain)
We merge datasets onto the skeleton using LEFT JOINs so the master table retains the full district–date grid.

In [ ]:
# 4.1 Left-join enrolment data

master_table = (
    skeleton
    .merge(
        enroll_dist,
        on=["date", "state", "district"],
        how="left"
    )
)


### 4.1 LEFT JOIN: enrolment
**What this cell does**
- Merges `enroll_dist` onto `skeleton` on `(date, state, district)`
- Uses `how="left"` so no skeleton rows are dropped
- Produces `master_table` (skeleton + enrolment measures)

In [ ]:
master_table.head()

,state,district,date,age_0_5,age_5_17,age_18_greater
0,100000,100000,2025-03-01,NaN,NaN,NaN
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,NaN,NaN,NaN
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,2025-03-01,NaN,NaN,NaN
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,2025-03-01,NaN,NaN,NaN
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,2025-03-01,NaN,NaN,NaN


In [ ]:
master_table.shape

(335070, 8)

In [ ]:
# 4.3 Left-join biometric update data

master_table = master_table.merge(
    bio_dist,
    on=["date", "state", "district"],
    how="left"
)


### 4.2 LEFT JOIN: biometric updates
**What this cell does**
- Merges `bio_dist` onto the existing `master_table`
- Keyed on `(date, state, district)` with `how="left"`
- Adds biometric measures without dropping any rows

In [ ]:
# 4.4 Validate join correctness

# Check 1: Row count preserved
assert master_table.shape[0] == skeleton.shape[0]

# Check 2: No duplicate keys introduced
assert master_table.duplicated(
    ["date", "state", "district"]
).sum() == 0

# Check 3: Column list sanity
master_table.columns.to_list()



['state',
 'district',
 'date',
 'age_0_5',
 'age_5_17',
 'age_18_greater',
 'demo_age_5_17',
 'demo_age_17_',
 'bio_age_5_17',
 'bio_age_17_']

### 4.3 Merge integrity checks
Asserts row preservation and key uniqueness; prints the final column list.

In [ ]:
enroll_start = enroll_dist["date"].max()
demo_start   = demo_dist["date"].max()
bio_start    = bio_dist["date"].max()

enroll_start, demo_start, bio_start

(Timestamp('2025-12-31 00:00:00'),
 Timestamp('2025-12-29 00:00:00'),
 Timestamp('2025-12-29 00:00:00'))

## 5. Temporal alignment + controlled filling
Goal: distinguish **true absence** (missing because data is not available) from **valid zeros** (after a defined global start date).
- Assumes `GLOBAL_START` and `ALL_MEASURE_COLS` are defined earlier in the notebook/pipeline.

In [ ]:
# 5.3 Apply zero-filling from the global start date
master_table.loc[
    master_table["date"] >= GLOBAL_START,
    ALL_MEASURE_COLS
] = (
    master_table.loc[
        master_table["date"] >= GLOBAL_START,
        ALL_MEASURE_COLS
    ]
    .fillna(0)
)


### 5.1 Apply zero fill on/after `GLOBAL_START`
**What this cell does**
- Restricts to rows with `date >= GLOBAL_START`
- Fills NaNs in `ALL_MEASURE_COLS` with 0
- Preserves NaNs *before* the global start date to avoid overstating earlier coverage

In [ ]:
# 5.5 add a single availability flag
master_table["data_available"] = master_table["date"] >= GLOBAL_START
master_table[master_table["data_available"]]

,state,district,date,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,data_available
0,100000,100000,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,True
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,2025-03-01,0.0,0.0,0.0,32.0,360.0,178.0,101.0,True
...,...,...,...,...,...,...,...,...,...,...,...
335065,WEST BENGAL,WEST MEDINIPUR,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
335066,WEST BENGAL,WEST MIDNAPORE,2025-12-31,22.0,20.0,1.0,0.0,0.0,0.0,0.0,True
335067,WEST BENGLI,HOOGHLY,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
335068,WESTBENGAL,HOOGHLY,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True


## End state
**Result**
- `master_table` is the district × date panel ready for feature engineering

**Expected next step (not executed here)**
- Export `master_table` to a final CSV and proceed to index construction